# Post-Training Data Collection

Notebook for building the 5-stage customer-service post-training dataset.
Stages: cold-start CoT SFT → general SFT → DPO → reward model → GRPO.

In [3]:
# Standard library imports
import json
import os
import random
import re
import time
from pathlib import Path
from pprint import pprint
from typing import Any, Dict, List, Optional, Tuple


In [8]:
# Third-party imports
import jsonlines
import numpy as np
import pandas as pd
import requests
from pydantic import BaseModel, Field, field_validator

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("jsonlines: imported")
print("requests:", requests.__version__)

pandas: 2.3.3
numpy: 2.2.6
jsonlines: imported
requests: 2.34.2


# Setting up schema for all 5 stages of the post-training pipeline

In [10]:
from pydantic import BaseModel, Field, field_validator
from typing import Any, Dict, List, Optional

## Stage 1: Class to handle data for Cold Start COT 

In [15]:
class ColdStartCoTSFTExample(BaseModel): 
    prompt: str = Field(..., description = "Customer Query") 
    response: str = Field(..., description = "Assistant response with <thinking> reasoning trace")

    @field_validator("response")
    @classmethod
    def must_contain_thinking(cls, v:str) -> str: 
        if "<thinking>" not in v or "</thinking>" not in v:
            raise ValueError("Response must contain <thinking>...</thinking> reasoning trace")
        return v

## Stage 2: Class to handle data for general SFT

In [18]:
class GeneralSFTExample(BaseModel): 
    prompt: str = Field(..., description = "Customer Query")
    response: str = Field(..., description = "Assistant response (no thinking trace required)")

## Stage 3: Class to handle data for DPO - Direct Preference Optimization

In [21]:
class DPOExample(BaseModel): 
    prompt: str = Field(..., description = "Customer Query")
    chosen: str = Field(..., description = "Assistant response (no thinking trace required)")
    rejected: str = Field(..., description = "A flawed response to contrast with chosen")

    @field_validator("chosen", "rejected")
    @classmethod
    def not_empty(cls, v:str) -> str:
        if not v.strip():
            raise ValueError("chosen and rejected must be non-empty")
        return v

## Stage 4: Class to handle data for Reward Model

In [24]:
class Rubric(BaseModel): 
    helpfulness: int = Field(...,ge=1, le=5)
    correctness: int = Field(..., ge=1, le=5)
    tone: int = Field(..., ge=1,le=5)


class RewardModelExample(BaseModel): 
    prompt: str = Field(..., description = "customer query")
    response: str = Field(..., description = "Candidate response to score")
    score: int = Field(..., ge=1, le=5, description = "Overall score between 1 and 5")
    rubric: Optional[Rubric] = Field(None, description = "Breakdown per dimension")

## Stage 5: Class to handle data for GRPO- Group Relative Policy Optimization

In [25]:
class GRPOExample(BaseModel): 
    prompt: str = Field(..., description = "Customer query")
    verifiable_reward: Dict[str,bool] = Field(..., 
                                              description = "Criteria dict with boolean checks, e.g. {'return_window_mentioned': True}")

    @field_validator("verifiable_reward")
    @classmethod
    def at_least_one_criterion(cls, v: Dict[str, bool]) -> Dict[str,bool]: 
        if len(v) == 0: 
            raise ValueError("Must have at least 1 verifiable criterion")
        return v
    

## Validator Script to run over the jsonl file

In [28]:
import jsonlines

def validate_jsonl(path: str, schema: type[BaseModel]) -> list[Dict]: 
    """
    Load and validate every line in the jsonl file against the Pydantic Schema. 
    Returns a list of all validation errors as dict : {line_number: errors}
    """

    errors = []
    with jsonlines.open(path) as reader: 
        for i, obj in enumerate(reader): 
            try: 
                schema(**obj)
            except Exception as e: 
                errors.append({i: str(e)})
    return errors
    

In [29]:
errs = validate_jsonl("data/01_cold_start_cot_sft/data.jsonl", ColdStartCoTSFTExample)
if not errs:
    print("All good")
else:
    for e in errs:
        print(e)

FileNotFoundError: [Errno 2] No such file or directory: 'data/01_cold_start_cot_sft/data.jsonl'

In [30]:
import os
# Set OPENROUTER_API_KEY in your environment or .env file before running

In [31]:
import json
import requests
import time

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"


In [32]:
def call_openrouter(model: str, system_prompt: str, user_prompt: str, n: int = 1) -> list[Dict]:
    """Call OpenRouter and return parsed json responses."""
    headers = {
       "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json", 
    }
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "n": n,
        "response_format": {"type": "json_object"},
    }
    resp = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=60)
    resp.raise_for_status()
    choices = resp.json()["choices"]
    results = []
    for choice in choices:
        content = choice["message"]["content"]
        results.append(json.loads(content))
    return results

In [35]:
DOMAIN = "retail customer service (order status, returns, refunds, promotions)"

STAGE_PROMPTS = {
    "01_cold_start_cot_sft": {
        "model": "deepseek/deepseek-v4-flash",
        "system": f"You generate training data for post-training LLMs in {DOMAIN}. "
                  "Return a JSON object with 'prompt' and 'response' keys. "
                  "The response MUST contain a <thinking>...</thinking> block showing "
                  "step-by-step reasoning (check policy, look up info, decide action), "
                  "followed by the customer-facing answer.",
        "user": "Generate one realistic customer service scenario. "
                "The customer asks about a return or refund. "
                "Show the agent's reasoning inside <thinking> tags, then give the final reply.",
        "schema": ColdStartCoTSFTExample,
    },
    "02_general_sft": {
        "model": "deepseek/deepseek-v4-flash",
        "system": f"You generate training data for post-training LLMs in {DOMAIN}. "
                  "Return a JSON object with 'prompt' and 'response' keys. "
                  "The response is a direct, helpful answer — no thinking trace.",
        "user": "Generate one realistic customer service Q&A about order status or promotions.",
        "schema": GeneralSFTExample,
    },
    "03_dpo": {
        "model": "deepseek/deepseek-v4-flash",
        "system": f"You generate preference training data for {DOMAIN}. "
                  "Return a JSON object with 'prompt', 'chosen', and 'rejected' keys. "
                  "'chosen' is an ideal response. 'rejected' is a flawed response "
                  "(too blunt, wrong policy, hallucinated info, or pushy tone).",
        "user": "Generate one customer service scenario with a good response and a bad response.",
        "schema": DPOExample,
    },
    "04_reward_model": {
        "model": "deepseek/deepseek-v4-flash",
        "system": f"You generate reward model training data for {DOMAIN}. "
                  "Return a JSON object with 'prompt', 'response', 'score' (1-5), "
                  "and 'rubric' (helpfulness, correctness, tone — each 1-5).",
        "user": "Generate one customer service response and score it on a 1-5 rubric.",
        "schema": RewardModelExample,
    },
    "05_grpo": {
        "model": "deepseek/deepseek-v4-flash",
        "system": f"You generate GRPO prompts with verifiable rewards for {DOMAIN}. "
                  "Return a JSON object with 'prompt' and 'verifiable_reward' keys. "
                  "'verifiable_reward' is a dict of boolean criteria that a correct "
                  "response should satisfy.",
        "user": "Generate one customer service prompt where the correct answer can be "
                "verified against specific criteria (e.g., return_window_mentioned, "
                "refund_method_correct, policy_adhered).",
        "schema": GRPOExample,
    },
}

## Generating 2 samples

In [37]:
N_PER_STAGE = 2

for stage_name, config in STAGE_PROMPTS.items():
    print(f"\n=== {stage_name} ===")
    results = call_openrouter(
        model=config["model"],
        system_prompt=config["system"],
        user_prompt=config["user"],
        n=N_PER_STAGE,
    )
    for i, obj in enumerate(results):
        try:
            validated = config["schema"](**obj)
            print(f"\n--- Sample {i+1}: VALID ---")
            print(validated.model_dump_json(indent=2))
        except Exception as e:
            print(f"\n--- Sample {i+1}: INVALID ---")
            print(f"Error: {e}")
            print(f"Raw: {json.dumps(obj, indent=2)}")


=== 01_cold_start_cot_sft ===

--- Sample 1: VALID ---
{
  "prompt": "Customer: I want to return a pair of jeans I bought last week, but I lost the receipt. Can I still get a refund?",
  "response": "<thinking>The customer wants to return jeans without a receipt. First, I need to check the store's return policy: we generally require a receipt or proof of purchase for refunds, but we can look up the order using their email or phone number. If they ordered online, I can find the order in the system. If they bought in-store, the transaction might be traceable if they used a credit card or loyalty program. Let me ask for their name, email, or the payment method used. Also, the jeans are from last week, so they are within the 30-day return window. If we can find the purchase, we can process a refund to the original payment method, or offer store credit if the receipt is missing. I'll proceed to try to locate the order.</thinking> I understand you'd like to return the jeans. No worries abou